# Capstone — Content Refresh Opportunity Scoring

**Lane:** Refresh / Content Opportunity Scoring

**Research question:** Can historical content and search-performance signals rank content items that are most likely to be associated with observed performance decline, so an editor can prioritise limited review capacity?

**Decision supported:** Which content items should an editor investigate first?

This notebook is the analysis behind the deployed research paper. It uses the anonymised FlyRank starter snapshot already used in W01–W04 and carries forward the completed data-contract, leakage, signal-audit, and baseline decisions.

## 1. Question

The unit of analysis is one pseudonymised content item. The output is a ranked review queue, not an automatic publishing decision. The main evaluation metric is Precision@50 because the practical use case is prioritisation at the top of a limited editorial queue.

**Target:** `is_declining_label = 1` when `trend_direction == 'down'`. This is an observed snapshot label, not a causal or future-refresh outcome.

**Important framing:** the starter snapshot does not establish that refreshing a page causes recovery, nor does it prove anything about Google's ranking algorithm.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Locate the repository root from either the repo root or work/notebooks/.
here = Path.cwd().resolve()
repo_root = next((p for p in [here, *here.parents] if (p / 'data/raw/content_refresh_anonymized.csv').exists()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate data/raw/content_refresh_anonymized.csv')
raw_path = repo_root / 'data/raw/content_refresh_anonymized.csv'
out_dir = repo_root / 'work/outputs'
out_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_path)
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)
print(f'Rows: {len(df):,}')
print(f'Clients: {df.client_id.nunique():,}')
print(f'Declining base rate: {df.is_declining_label.mean():.1%}')

## 2. Data

**Release:** the anonymised FlyRank starter snapshot bundled in `data/raw/content_refresh_anonymized.csv`.

**Scale:** 30,000 content items across 32 pseudonymised clients.

**Public-safe exclusions:** client identifiers are used only for grouped splitting; content identifiers are used only to join/display anonymous queue rows; no client names, domains, URLs, private queries, credentials, or raw exports are published.

**Leakage exclusions:** `trend_direction`, `trend_pct`, the derived target, and any target-derived product flags are excluded from the feature matrix.

In [ ]:
# Candidate features carried forward from the W03 contract / reference ML pipeline.
NUMERIC_FEATURES = [
    'search_volume','competition','cpc','word_count','char_count',
    'impressions_90d','clicks_90d','sessions_90d','days_with_impressions',
    'days_with_sessions','content_age_days','days_since_last_update',
    'ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct'
]
CATEGORICAL_FEATURES = [
    'competition_level','content_type','main_intent','age_tier',
    'freshness_tier','word_count_tier','impression_tier','position_tier'
]

missing = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c not in df.columns]
if missing:
    raise ValueError(f'Missing contracted features: {missing}')

print('Feature columns:', len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES))
print('Excluded from model:', ['content_id','client_id','trend_direction','trend_pct','is_declining_label'])

## 3. Methodology

The model uses only fields available in the starter snapshot and excludes the observed trend label from the feature matrix. Numeric missing values are coerced and filled with zero; categorical missing values become `unknown`. Categorical fields are one-hot encoded.

Validation uses a **client holdout**: approximately 20% of pseudonymised clients are held out, so content from those clients is not mixed into training. This is the same split strategy used by the existing repository model report.

The transparent W04 baseline is: `stale_flag × visible_flag × log1p(impressions_90d)`, where stale means at least 180 days since update and visible means at least 300 impressions in 90 days. The baseline is scored without the target and evaluated on the same holdout.

In [ ]:
RANDOM_STATE = 42

# Build model matrix.
num = df[NUMERIC_FEATURES].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
cat = df[CATEGORICAL_FEATURES].fillna('unknown').astype(str)
cat_encoded = pd.get_dummies(cat, prefix=CATEGORICAL_FEATURES, dtype=float)
X = pd.concat([num.reset_index(drop=True), cat_encoded.reset_index(drop=True)], axis=1)
y = df['is_declining_label'].astype(int)

# Client-aware holdout.
clients = df['client_id'].fillna('unknown').astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled) * 0.20)))
test_clients = set(shuffled[:test_client_count])
test_mask = clients.isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]
if y.iloc[train_idx].nunique() < 2 or y.iloc[test_idx].nunique() < 2:
    raise ValueError('Client holdout does not contain both target classes.')

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
print(f'Train rows: {len(train_idx):,}')
print(f'Test rows: {len(test_idx):,}')
print(f'Test clients: {len(test_clients):,}')

### Leakage check

The following fields are deliberately not present in `X`: `trend_direction`, `trend_pct`, `is_declining_label`, `content_id`, and `client_id`. The first three can encode the outcome; the IDs are identifiers/grouping variables rather than predictive signals.

This notebook therefore measures the model against the target only after the score has been generated.

In [ ]:
for forbidden in ['trend_direction','trend_pct','is_declining_label','content_id','client_id']:
    assert forbidden not in X.columns, f'Leakage/ID column entered model: {forbidden}'
print('Leakage/ID assertions passed.')

## 4. Baseline

The W04 rule prioritises stale pages with meaningful search visibility. It is intentionally simple and unfitted so an editor can understand why a row entered the queue.

In [ ]:
df['baseline_score'] = (
    (df['days_since_last_update'] >= 180).astype(int)
    * (df['impressions_90d'] >= 300).astype(int)
    * np.log1p(pd.to_numeric(df['impressions_90d'], errors='coerce').fillna(0))
)

def precision_at_k(y_true, scores, k):
    tmp = pd.DataFrame({'y': np.asarray(y_true), 'score': np.asarray(scores)})
    return float(tmp.sort_values('score', ascending=False).head(min(k, len(tmp)))['y'].mean())

baseline_test_scores = df.iloc[test_idx]['baseline_score'].to_numpy()
for k in [10,25,50,100]:
    print(f'Baseline Precision@{k}: {precision_at_k(y_test, baseline_test_scores, k):.3f}')
print(f'Baseline PR-AUC: {average_precision_score(y_test, baseline_test_scores):.3f}')

## 5. Model

The primary model is a Random Forest classifier because the task is tabular, mixed-type, nonlinear ranking. Logistic Regression is included as an interpretable benchmark. Model selection is based on Precision@50 on the client-held-out test set.

In [ ]:
models = {
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'random_forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, min_samples_leaf=25,
        class_weight='balanced_subsample', n_jobs=-1, random_state=RANDOM_STATE
    )
}

results = []
probabilities = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:,1]
    probabilities[name] = p
    results.append({
        'model': name,
        'ROC_AUC': roc_auc_score(y_test, p),
        'PR_AUC': average_precision_score(y_test, p),
        'Precision@10': precision_at_k(y_test, p, 10),
        'Precision@25': precision_at_k(y_test, p, 25),
        'Precision@50': precision_at_k(y_test, p, 50),
        'Precision@100': precision_at_k(y_test, p, 100),
        'Recall': recall_score(y_test, p >= 0.5, zero_division=0),
        'F1': f1_score(y_test, p >= 0.5, zero_division=0)
    })

results.append({
    'model': 'baseline_rules',
    'ROC_AUC': roc_auc_score(y_test, baseline_test_scores),
    'PR_AUC': average_precision_score(y_test, baseline_test_scores),
    'Precision@10': precision_at_k(y_test, baseline_test_scores, 10),
    'Precision@25': precision_at_k(y_test, baseline_test_scores, 25),
    'Precision@50': precision_at_k(y_test, baseline_test_scores, 50),
    'Precision@100': precision_at_k(y_test, baseline_test_scores, 100),
    'Recall': np.nan, 'F1': np.nan
})

results_df = pd.DataFrame(results).sort_values(['Precision@50','PR_AUC'], ascending=False)
results_df

## 6. Results

The table below compares every approach on the same held-out client set. The model is considered useful only if it improves the top-of-queue ranking over the transparent baseline.

In [ ]:
best_model_name = results_df[results_df.model != 'baseline_rules'].iloc[0]['model']
best_p = probabilities[best_model_name]
print('Selected model:', best_model_name)
print(results_df.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

# Persist reproducibility receipt.
results_df.to_csv(out_dir / 'capstone_model_results.csv', index=False)
with open(out_dir / 'capstone_metrics.json', 'w', encoding='utf-8') as f:
    json.dump({
        'split_strategy': 'client_holdout',
        'test_clients': sorted(test_clients),
        'best_model': best_model_name,
        'metrics': results_df.to_dict(orient='records')
    }, f, indent=2, default=lambda x: float(x) if isinstance(x, (np.floating,)) else x)

In [ ]:
# Model-vs-baseline Precision@K chart.
ks = [10,25,50,100]
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(ks, [results_df.loc[results_df.model=='baseline_rules', f'Precision@{k}'].iloc[0] for k in ks], marker='o', label='Baseline')
ax.plot(ks, [results_df.loc[results_df.model==best_model_name, f'Precision@{k}'].iloc[0] for k in ks], marker='o', label=best_model_name)
ax.set_xlabel('K')
ax.set_ylabel('Precision@K')
ax.set_title('Top-of-queue ranking performance')
ax.legend()
ax.grid(alpha=.25)
plt.tight_layout()
plt.savefig(out_dir / 'capstone_precision_at_k.png', dpi=160)
plt.show()

## 7. Error analysis

A false positive is a page the model prioritises that is not labelled declining. A false negative is a declining page that receives a lower score. These errors matter because the system is a reviewer aid: false positives consume editorial capacity, while false negatives can leave genuinely declining items unreviewed.

In [ ]:
error_frame = df.iloc[test_idx][['content_id','client_id','is_declining_label','impressions_90d','days_since_last_update','ctr','avg_position']].copy()
error_frame['model_probability'] = best_p
error_frame['predicted_decline'] = (best_p >= 0.5).astype(int)
error_frame['error_type'] = np.select([
    (error_frame.predicted_decline.eq(1) & error_frame.is_declining_label.eq(0)),
    (error_frame.predicted_decline.eq(0) & error_frame.is_declining_label.eq(1))
], ['false_positive','false_negative'], default='correct')
print(error_frame.error_type.value_counts().to_string())

print('Top false positives:')
display(error_frame[error_frame.error_type=='false_positive'].sort_values('model_probability', ascending=False).head(10))
print('Top false negatives:')
display(error_frame[error_frame.error_type=='false_negative'].sort_values('model_probability').head(10))

## 8. Ranked recommendations

The final queue is generated by fitting the selected model on the complete starter snapshot after the held-out evaluation is complete. This produces a practical scoring queue while keeping the reported metrics based on the untouched client holdout.

Actions are decision-support labels, not causal prescriptions.

In [ ]:
final_model = models[best_model_name]
final_model.fit(X, y)
df['model_risk_score'] = final_model.predict_proba(X)[:,1]

# Reason codes use non-target signals only.
df['reason_code'] = np.select([
    (df.model_risk_score >= 0.75) & (df.impressions_90d >= 300) & (df.days_since_last_update >= 180),
    (df.model_risk_score >= 0.75) & (df.impressions_90d >= 300),
    (df.model_risk_score >= 0.75),
    (df.impressions_90d >= 300) & (df.days_since_last_update >= 180)
], [
    'model_risk_stale_visible',
    'model_risk_visible',
    'model_risk',
    'stale_visible_baseline'
], default='monitor')

df['action'] = np.select([
    df.model_risk_score >= 0.75,
    (df.model_risk_score >= 0.50) & (df.impressions_90d >= 300),
    df.model_risk_score >= 0.50
], ['refresh_review','priority_review','monitor'], default='monitor')

queue = df[['content_id','model_risk_score','action','reason_code','impressions_90d','days_since_last_update','ctr','avg_position']].copy()
queue = queue.sort_values('model_risk_score', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', np.arange(1, len(queue)+1))
queue.to_csv(out_dir / 'capstone_ranked_recommendations.csv', index=False)

print('Top 20 anonymised recommendations:')
display(queue.head(20))

In [ ]:
# Feature importance for the selected model.
if best_model_name == 'random_forest':
    importance = pd.DataFrame({'feature': X.columns, 'importance': final_model.feature_importances_}).sort_values('importance', ascending=False).head(15)
    display(importance)
    fig, ax = plt.subplots(figsize=(8,6))
    ax.barh(importance.feature[::-1], importance.importance[::-1])
    ax.set_title('Top model features')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(out_dir / 'capstone_feature_importance.png', dpi=160)
    plt.show()

## 9. Limitations and honest framing

- The starter snapshot is cross-sectional for this analysis; the label represents an observed trend category, not a clean future-window outcome.
- The model identifies associations useful for prioritisation; it does not establish that refreshing a page causes recovery.
- The data cannot prove or explain Google's ranking algorithm.
- Client grouping improves separation, but the sample is still an anonymised research/education snapshot.
- Thresholds such as 180 days, 300 impressions, and model-risk cutoffs are operational choices, not universal rules.
- Ranked recommendations should be manually reviewed against editorial context before action.

## 10. Five-sentence abstract

This study asks whether historical content and search-performance signals can rank content items for refresh review. Using 30,000 anonymised content items from the FlyRank starter snapshot, it compares a transparent stale-and-visible baseline with supervised tabular models using leakage-controlled features. Models are evaluated on a client-held-out test set with Precision@K and average precision, with Precision@50 as the primary prioritisation metric. The selected model and its measured improvement or shortfall versus the baseline are reported directly from the reproducibility table above. The resulting ranked queue is intended as decision support for editorial review, not as evidence of causal refresh impact or Google's ranking behaviour.

## 11. Ranked action playbook

1. **Refresh review:** highest-risk items should be inspected first, especially when they also have meaningful search visibility.
2. **Priority review:** medium/high model risk with visible search activity merits manual investigation.
3. **Monitor:** lower-risk items remain in the queue for observation rather than automatic intervention.

The reason codes make each recommendation auditable from non-target signals.

## 12. Reproducibility artifacts

This notebook writes:

- `work/outputs/capstone_model_results.csv` — model-vs-baseline metrics.
- `work/outputs/capstone_metrics.json` — split and metric receipt.
- `work/outputs/capstone_ranked_recommendations.csv` — anonymised ranked queue.
- `work/outputs/capstone_precision_at_k.png` — ranking comparison chart.
- `work/outputs/capstone_feature_importance.png` — feature-importance chart when Random Forest is selected.

No raw client names, domains, URLs, private queries, credentials, or raw warehouse exports are written by this notebook.

## 13. 5-minute demo outline

**0:00–0:45:** State the decision: which content should an editor review first?

**0:45–1:30:** Show the anonymised dataset, target, and leakage exclusions.

**1:30–2:30:** Explain the W04 transparent baseline and the client-held-out model comparison.

**2:30–3:30:** Show Precision@K and the model-vs-baseline result.

**3:30–4:30:** Show the ranked queue and reason codes.

**4:30–5:00:** State limitations and the decision-support framing.

## 14. Employer-facing summary

I built a leakage-controlled content-refresh prioritisation model on anonymised search-performance data. The system compares a transparent editorial baseline with supervised tabular ML using a client-held-out evaluation and top-of-queue ranking metrics. The final output is an auditable ranked review queue with reason codes, designed for decision support rather than automatic publishing or causal claims.

## Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset**. Data credit: [FlyRank](https://flyrank.ai).

This public-facing work follows the internship's public-safe rule: no client names, domains, URLs, private queries, credentials, or raw exports are published.

## Self-check

- [ ] Every section is filled and backed by executable code where applicable.
- [ ] The notebook runs top-to-bottom without errors.
- [ ] The same client-held-out split is used for baseline and models.
- [ ] Target-derived fields are excluded from the feature matrix.
- [ ] Metrics are measured, not invented.
- [ ] Claims use observed/measured/directional/decision-support language.
- [ ] Public output contains no client names, domains, URLs, private queries, credentials, or raw exports.
- [ ] Paper URL is placed only in `submission/paper_url.txt` once the paper is deployed.